# 07 — Entraînement et évaluation des modèles

## A — Introduction et objectifs

Ce notebook constitue la première phase d'entraînement des modèles de Machine Learning du projet **Smart Energy Optimizer**.

Les étapes précédentes ont permis de préparer un dataset supervisé, de définir un protocole de validation chronologique et d'évaluer deux méthodes de prévision naïves. Les résultats ont notamment mis en évidence une forte périodicité hebdomadaire dans les consommations énergétiques simulées.

L'objectif est désormais d'expérimenter plusieurs algorithmes de régression afin de déterminer leur capacité à prévoir la consommation électrique des sites sur des horizons compris entre **H+1 et H+24**.

### Objectifs

- Entraîner un premier modèle de régression linéaire.
- Expérimenter des modèles non linéaires, notamment Random Forest.
- Évaluer les prédictions à l'aide des métriques MAE et RMSE.
- Comparer les résultats aux baselines journalière et hebdomadaire.
- Analyser les performances selon les types de sites et les horizons de prévision.
- Assurer la traçabilité des expérimentations avec MLflow.

### Méthodologie

Les modèles seront entraînés à partir du dataset préparé lors des étapes précédentes, en conservant la séparation chronologique définie dans le notebook `06_experimentation_ml.ipynb`.

Le prétraitement sera intégré aux pipelines scikit-learn afin de garantir la cohérence des transformations appliquées pendant l'entraînement et la prédiction.

Les expériences seront réalisées dans ce notebook avant que les traitements retenus soient progressivement transférés vers des modules Python réutilisables dans `src/smart_energy_ml/`.

### Critère de réussite

Notre référence principale sera la **baseline hebdomadaire H-168**, qui a obtenu une MAE de **6,78 kW** et une RMSE de **10,48 kW** sur les données de validation.

L'objectif ne sera pas uniquement d'entraîner un modèle, mais de déterminer si les algorithmes expérimentés apportent une amélioration mesurable par rapport à cette méthode de référence.

### B - Chargement des données

In [198]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../data/processed")

df = pd.read_parquet(DATA_DIR / "training_dataset.parquet")

print("Dimensions du dataset :", df.shape)
print("Nombre de sites :", df["site_id"].nunique())
print("Horizons disponibles :", sorted(df["horizon"].unique()))

display(df.head())

Dimensions du dataset : (118860, 14)
Nombre de sites : 7
Horizons disponibles : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24)]


,forecast_origin,target_timestamp,site_id,site_type,horizon,target_hour,target_day_of_week,target_month,target_is_weekend,lag_1h,lag_24h,rolling_mean_24h,target_lag_168h,target_consumption_kw
0,2026-09-24 00:00:00+00:00,2026-09-24 01:00:00+00:00,SITE001,office,1,3,3,9,0,NaN,NaN,NaN,NaN,62.300833
1,2026-09-24 01:00:00+00:00,2026-09-24 02:00:00+00:00,SITE001,office,1,4,3,9,0,60.726364,NaN,NaN,NaN,62.099000
2,2026-09-24 02:00:00+00:00,2026-09-24 03:00:00+00:00,SITE001,office,1,5,3,9,0,62.300833,NaN,NaN,NaN,61.398182
3,2026-09-24 03:00:00+00:00,2026-09-24 04:00:00+00:00,SITE001,office,1,6,3,9,0,62.099000,NaN,NaN,NaN,60.970909
4,2026-09-24 04:00:00+00:00,2026-09-24 05:00:00+00:00,SITE001,office,1,7,3,9,0,61.398182,NaN,NaN,NaN,62.373333


### C - Séparation des données d'entrainement et de validation

#### C.1 — Constitution des ensembles d'entraînement et de validation
Afin de garantir la comparabilité des expérimentations, nous conservons la date de séparation définie dans le notebook précédent, fixée au 15 octobre 2026.

Les exemples d'entraînement doivent disposer d'une consommation cible antérieure à cette date. Les exemples de validation doivent, quant à eux, avoir une origine de prévision postérieure ou égale à cette date.

Les observations situées à cheval sur la séparation sont exclues afin d'éviter qu'une cible appartenant à la période de validation soit utilisée pendant l'entraînement.

In [199]:
SPLIT_DATE = pd.Timestamp("2026-10-15", tz="UTC")

# Entraînement : seules les cibles antérieures à la séparation
df_train = df.loc[df["target_timestamp"] < SPLIT_DATE].copy()

# Validation : seules les prévisions réalisées à partir de la séparation
df_val = df.loc[df["forecast_origin"] >= SPLIT_DATE].copy()

print("Exemples d'entraînement :", len(df_train))
print("Exemples de validation :", len(df_val))
print("Exemples exclus :", len(df) - len(df_train) - len(df_val))

# Vérification de l'intégrité temporelle
assert df_train["target_timestamp"].max() < SPLIT_DATE
assert df_val["forecast_origin"].min() >= SPLIT_DATE

Exemples d'entraînement : 82572
Exemples de validation : 34188
Exemples exclus : 2100


#### C.2 — Sélection des variables explicatives et de la cible

L'entraînement supervisé nécessite de distinguer les variables explicatives (*features*) de la variable cible (*target*).

Les variables explicatives regroupent les caractéristiques du site, les informations calendaires, l'horizon de prévision et les consommations historiques disponibles au moment de la prédiction.

La variable cible correspond à la consommation électrique future, exprimée en kilowatts (`target_consumption_kw`).

Nous conservons les mêmes variables que celles définies lors de la phase d'expérimentation, notamment `target_lag_168h`, afin d'exploiter la périodicité hebdomadaire identifiée lors de l'évaluation des baselines.

Les colonnes `forecast_origin` et `target_timestamp` sont conservées dans les datasets de référence pour les contrôles temporels, mais ne sont pas directement transmises au modèle.

In [200]:
feature_columns = [
    "site_id",
    "site_type",
    "horizon",
    "target_hour",
    "target_day_of_week",
    "target_month",
    "target_is_weekend",
    "lag_1h",
    "lag_24h",
    "rolling_mean_24h",
    "target_lag_168h",
]

target_column = "target_consumption_kw"

# Variables explicatives
X_train = df_train[feature_columns].copy()
X_val = df_val[feature_columns].copy()

# Valeurs réelles à prédire
y_train = df_train[target_column].copy()
y_val = df_val[target_column].copy()

print("X_train :", X_train.shape)
print("X_val   :", X_val.shape)
print("y_train :", y_train.shape)
print("y_val   :", y_val.shape)

display(X_train.head())

X_train : (82572, 11)
X_val   : (34188, 11)
y_train : (82572,)
y_val   : (34188,)


,site_id,site_type,horizon,target_hour,target_day_of_week,target_month,target_is_weekend,lag_1h,lag_24h,rolling_mean_24h,target_lag_168h
0,SITE001,office,1,3,3,9,0,NaN,NaN,NaN,NaN
1,SITE001,office,1,4,3,9,0,60.726364,NaN,NaN,NaN
2,SITE001,office,1,5,3,9,0,62.300833,NaN,NaN,NaN
3,SITE001,office,1,6,3,9,0,62.099000,NaN,NaN,NaN
4,SITE001,office,1,7,3,9,0,61.398182,NaN,NaN,NaN


### D.1 — Construction du pipeline de régression linéaire

Notre premier modèle repose sur une régression linéaire, un algorithme supervisé qui cherche à établir une relation entre les variables explicatives et la consommation électrique à prédire.

Avant l'entraînement, les données nécessitent plusieurs transformations :

- Les variables numériques et calendaires sont conservées sous forme numérique.
- Les consommations historiques manquantes sont remplacées par la médiane calculée sur l'ensemble d'entraînement.
- Les variables catégorielles (`site_id`, `site_type`) sont transformées par encodage One-Hot.

Ces opérations sont regroupées dans un `ColumnTransformer`, lui-même intégré à un `Pipeline` scikit-learn avec le modèle de régression.

Cette approche permet d'automatiser les transformations, de limiter les risques de fuite de données et de préparer la future industrialisation du modèle.

In [201]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Variables numériques et calendaires
numeric_columns = [
    "horizon",
    "target_hour",
    "target_day_of_week",
    "target_month",
    "target_is_weekend",
]

# Consommations historiques
historical_columns = [
    "lag_1h",
    "lag_24h",
    "rolling_mean_24h",
    "target_lag_168h",
]

# Variables catégorielles
categorical_columns = [
    "site_id",
    "site_type",
]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", "passthrough", numeric_columns),
        (
            "historical",
            SimpleImputer(strategy="median"),
            historical_columns,
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_columns,
        ),
    ]
)

linear_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression()),
    ]
)

print(linear_pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric', 'passthrough',
                                                  ['horizon', 'target_hour',
                                                   'target_day_of_week',
                                                   'target_month',
                                                   'target_is_weekend']),
                                                 ('historical',
                                                  SimpleImputer(strategy='median'),
                                                  ['lag_1h', 'lag_24h',
                                                   'rolling_mean_24h',
                                                   'target_lag_168h']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['site_id', 'site_type'])])),
  

#### D.2 — Entraînement de la régression linéaire
Nous procédons au premier entraînement supervisé à partir des 82 572 exemples constituant notre ensemble d'apprentissage.

L'appel à fit() déclenche successivement le prétraitement des données et l'ajustement des coefficients de la régression linéaire.

Une fois entraîné, le pipeline peut produire des prédictions sur l'ensemble de validation, sans réapprendre les transformations à partir de ces nouvelles observations.

In [202]:
# Entraînement du pipeline complet
linear_pipeline.fit(X_train, y_train)

# Prédictions sur les données de validation
y_pred_linear = linear_pipeline.predict(X_val)

print("Entraînement terminé !")
print("Nombre de prédictions :", len(y_pred_linear))

# Comparaison des premières prédictions avec les valeurs réelles
df_predictions = pd.DataFrame(
    {
        "reel_kw": y_val.to_numpy(),
        "prediction_kw": y_pred_linear,
    }
)

display(df_predictions.head(10))

Entraînement terminé !
Nombre de prédictions : 34188


,reel_kw,prediction_kw
0,61.848000,4.421838
1,62.279167,9.563844
2,62.126667,14.425295
3,61.062500,19.041049
4,62.819167,24.732738
5,62.744167,29.671836
6,62.701667,35.918294
7,83.013333,53.016801
8,126.590000,76.699384
9,169.975833,99.646284


### D.3 — Évaluation des performances

Après l'entraînement, nous évaluons les performances de notre premier modèle sur l'ensemble de validation.

Deux métriques sont utilisées :

- **MAE (Mean Absolute Error)** : moyenne des écarts absolus entre les consommations prédites et les consommations réelles.
- **RMSE (Root Mean Squared Error)** : racine carrée de la moyenne des erreurs quadratiques, accordant davantage de poids aux erreurs importantes.

Ces métriques sont exprimées en kilowatts et permettent de comparer directement les résultats du modèle avec ceux des baselines définies précédemment.

L'objectif est de déterminer si la régression linéaire apporte une amélioration par rapport aux méthodes de prévision naïves.

In [203]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Calcul des métriques
linear_mae = mean_absolute_error(y_val, y_pred_linear)
linear_rmse = np.sqrt(mean_squared_error(y_val, y_pred_linear))

# Comparaison avec les baselines du notebook 05
results = pd.DataFrame(
    [
        {
            "modele": "Baseline journalière H-24",
            "MAE": 36.49,
            "RMSE": 114.25,
        },
        {
            "modele": "Baseline hebdomadaire H-168",
            "MAE": 6.78,
            "RMSE": 10.48,
        },
        {
            "modele": "Régression linéaire",
            "MAE": linear_mae,
            "RMSE": linear_rmse,
        },
    ]
)

display(results.round(2))

,modele,MAE,RMSE
0,Baseline journalière H-24,36.49,114.25
1,Baseline hebdomadaire H-168,6.78,10.48
2,Régression linéaire,64.22,77.72


#### D.4 — Analyse des coefficients de la régression linéaire globale

Cette étape vise à examiner les coefficients appris par notre premier modèle afin de mieux comprendre son comportement et les raisons possibles de ses performances limitées.

Nous cherchons notamment à déterminer comment la régression combine les caractéristiques des sites, les variables calendaires et les consommations historiques.

Une attention particulière est portée à `target_lag_168h`, dont la pertinence a été mise en évidence lors de l'évaluation de la baseline hebdomadaire.

Les coefficients doivent toutefois être interprétés avec prudence, notamment en raison des différences d'échelle entre les variables et des dépendances entre les caractéristiques catégorielles.

In [204]:
# Récupération des noms des variables après transformation
feature_names = linear_pipeline.named_steps["preprocessor"].get_feature_names_out()

# Coefficients appris par la régression linéaire
coefficients = linear_pipeline.named_steps["model"].coef_

df_coefficients = pd.DataFrame(
    {
        "variable": feature_names,
        "coefficient": coefficients,
    }
)

display(
    df_coefficients.sort_values(
        "coefficient",
        key=abs,
        ascending=False,
    )
)

,variable,coefficient
19,categorical__site_type_office,-131.356221
11,categorical__site_id_SITE003,102.537871
16,categorical__site_type_datacenter,102.537871
14,categorical__site_id_SITE006,-69.346948
9,categorical__site_id_SITE001,-62.009273
17,categorical__site_type_factory,58.329450
20,categorical__site_type_retail,-51.692792
12,categorical__site_id_SITE004,-51.692792
4,numeric__target_is_weekend,-50.308029
10,categorical__site_id_SITE002,40.955630


#### D.5 — Régression linéaire sur la seule référence hebdomadaire

In [205]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

weekly_linear_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LinearRegression()),
    ]
)

# Entraînement uniquement sur la référence hebdomadaire
weekly_linear_pipeline.fit(
    X_train[["target_lag_168h"]],
    y_train,
)

# Prédictions
y_pred_weekly_linear = weekly_linear_pipeline.predict(X_val[["target_lag_168h"]])

# Évaluation
weekly_linear_mae = mean_absolute_error(y_val, y_pred_weekly_linear)

weekly_linear_rmse = np.sqrt(mean_squared_error(y_val, y_pred_weekly_linear))

print(f"MAE  : {weekly_linear_mae:.2f} kW")
print(f"RMSE : {weekly_linear_rmse:.2f} kW")

print(
    "Coefficient hebdomadaire :",
    weekly_linear_pipeline.named_steps["model"].coef_[0],
)

MAE  : 20.76 kW
RMSE : 22.53 kW
Coefficient hebdomadaire : 0.9920435942972226


#### D.6 — Diagnostic de l'impact des valeurs manquantes

In [206]:
# Identifier les observations disposant d'une référence hebdomadaire
mask = X_val["target_lag_168h"].notna()

print("Références disponibles :", mask.sum())
print("Références manquantes :", (~mask).sum())

# Évaluer uniquement les observations avec référence disponible
mae_available = mean_absolute_error(
    y_val.loc[mask],
    y_pred_weekly_linear[mask.to_numpy()],
)

print(f"MAE sans valeurs manquantes : {mae_available:.2f} kW")

# Vérifier l'interception de la régression
print(
    "Intercept :",
    weekly_linear_pipeline.named_steps["model"].intercept_,
)

Références disponibles : 34188
Références manquantes : 0
MAE sans valeurs manquantes : 20.76 kW
Intercept : 23.224795205395424


#### D.7 — Régression hebdomadaire sans imputation

In [207]:
# Conserver uniquement les exemples d'entraînement
# disposant d'une référence hebdomadaire réelle
train_mask = X_train["target_lag_168h"].notna()

X_train_weekly = X_train.loc[train_mask, ["target_lag_168h"]]

y_train_weekly = y_train.loc[train_mask]

# Nouvelle régression sans imputation
weekly_linear_clean = LinearRegression()

weekly_linear_clean.fit(
    X_train_weekly,
    y_train_weekly,
)

# Prédictions sur la même validation
y_pred_weekly_clean = weekly_linear_clean.predict(X_val[["target_lag_168h"]])

print("Exemples conservés :", len(X_train_weekly))

print(
    "Coefficient :",
    weekly_linear_clean.coef_[0],
)

print(
    "Intercept :",
    weekly_linear_clean.intercept_,
)

print(
    "MAE :",
    round(mean_absolute_error(y_val, y_pred_weekly_clean), 2),
)

print(
    "RMSE :",
    round(np.sqrt(mean_squared_error(y_val, y_pred_weekly_clean)), 2),
)

Exemples conservés : 56448
Coefficient : 1.0061514830159783
Intercept : -0.2354000261769329
MAE : 7.0
RMSE : 10.8


#### D.8 — Réentraînement du modèle global sur les données filtrées

Les expérimentations précédentes ont mis en évidence une dégradation importante des performances lorsque les références hebdomadaires manquantes étaient remplacées par la médiane.

En excluant ces observations, la régression linéaire utilisant uniquement `target_lag_168h` a obtenu une MAE de **7,00 kW**, contre **20,76 kW** avec imputation.

Nous cherchons désormais à déterminer si cette amélioration se retrouve sur notre modèle global, qui exploite l'ensemble des variables explicatives.

Pour cela, nous conservons uniquement les exemples d'entraînement disposant d'une référence hebdomadaire réelle, sans modifier notre ensemble de validation.

Cette expérience permettra d'évaluer si les mauvaises performances initiales provenaient principalement de l'imputation ou également des limites de la régression linéaire face à la diversité des comportements énergétiques.

In [208]:
from sklearn.base import clone

# Sélection des observations avec une référence hebdomadaire réelle
train_mask = X_train["target_lag_168h"].notna()

X_train_filtered = X_train.loc[train_mask].copy()
y_train_filtered = y_train.loc[train_mask].copy()

# Nouveau pipeline indépendant du modèle initial
linear_pipeline_filtered = clone(linear_pipeline)

# Entraînement sur les données filtrées
linear_pipeline_filtered.fit(
    X_train_filtered,
    y_train_filtered,
)

# Prédictions sur le même ensemble de validation
y_pred_linear_filtered = linear_pipeline_filtered.predict(X_val)

# Calcul des métriques
linear_filtered_mae = mean_absolute_error(
    y_val,
    y_pred_linear_filtered,
)

linear_filtered_rmse = np.sqrt(mean_squared_error(y_val, y_pred_linear_filtered))

print("Exemples d'entraînement :", len(X_train_filtered))
print(f"MAE  : {linear_filtered_mae:.2f} kW")
print(f"RMSE : {linear_filtered_rmse:.2f} kW")

Exemples d'entraînement : 56448
MAE  : 7.04 kW
RMSE : 10.87 kW


#### D.9 — Bilan des expérimentations sur la régression linéaire

Cette première série d'expérimentations a permis d'entraîner un modèle de régression linéaire et d'analyser l'influence du prétraitement sur ses performances.

Le modèle initial, entraîné sur l'ensemble des données avec imputation médiane des valeurs historiques manquantes, obtenait une **MAE de 64,22 kW** et une **RMSE de 77,72 kW**, soit des résultats nettement inférieurs à ceux de notre baseline hebdomadaire.

L'analyse des coefficients et plusieurs expérimentations complémentaires ont permis d'identifier une cause majeure de cette dégradation : l'imputation des références hebdomadaires manquantes introduisait des valeurs artificielles peu représentatives des consommations réelles.

En excluant les observations ne disposant pas de référence hebdomadaire, nous avons obtenu les résultats suivants :

| Modèle | MAE | RMSE |
|---|---:|---:|
| Régression linéaire initiale | 64,22 kW | 77,72 kW |
| Régression hebdomadaire filtrée | 7,00 kW | 10,80 kW |
| Régression globale filtrée | 7,04 kW | 10,87 kW |
| Baseline hebdomadaire H-168 | **6,78 kW** | **10,48 kW** |

Ces résultats mettent en évidence plusieurs enseignements :

- **La qualité et la disponibilité des variables explicatives sont déterminantes.** Une stratégie d'imputation inadaptée peut dégrader fortement l'apprentissage.
- **La référence hebdomadaire constitue un signal prédictif majeur** sur notre jeu de données synthétiques.
- **La complexité des variables utilisées ne garantit pas une amélioration des performances.** Notre régression globale n'apporte pas, à ce stade, de gain par rapport au modèle utilisant uniquement la référence hebdomadaire.
- **Une baseline simple reste difficile à surpasser** lorsque les données présentent une forte régularité temporelle.

Ces résultats restent propres au jeu de données étudié, qui couvre une période limitée et repose sur des consommations simulées.

La prochaine étape consistera à expérimenter un modèle **Random Forest Regressor**, capable de représenter des relations non linéaires et des interactions entre les variables.

L'objectif sera de déterminer si cette approche permet d'améliorer les prévisions, notamment en exploitant les différences de comportement entre les sites et les informations calendaires.

### E.1 — Construction du pipeline Random Forest

Afin d'étudier l'intérêt d'un algorithme non linéaire, nous expérimentons un modèle de type *Random Forest Regressor*.

Cet algorithme repose sur un ensemble d'arbres de décision entraînés sur différentes sélections d'observations et de variables. Leurs prédictions sont ensuite agrégées afin de produire une estimation finale.

Nous conservons les mêmes variables explicatives, le même prétraitement et les mêmes ensembles de données que lors de la régression linéaire globale filtrée, afin de garantir la comparabilité des résultats.

Pour cette première expérience, nous utilisons une configuration volontairement simple. L'optimisation des hyperparamètres sera envisagée après l'évaluation initiale.

In [209]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", clone(preprocessor)),
        (
            "model",
            RandomForestRegressor(
                n_estimators=100,
                max_depth=15,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

print(random_forest_pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric', 'passthrough',
                                                  ['horizon', 'target_hour',
                                                   'target_day_of_week',
                                                   'target_month',
                                                   'target_is_weekend']),
                                                 ('historical',
                                                  SimpleImputer(strategy='median'),
                                                  ['lag_1h', 'lag_24h',
                                                   'rolling_mean_24h',
                                                   'target_lag_168h']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['site_id', 'site_type'])])),
  

#### E.2 — Entraînement et évaluation du modèle Random Forest

Nous procédons à l'entraînement du modèle Random Forest sur les **56 448 observations** disposant d'une référence hebdomadaire valide.

Afin de garantir la comparabilité des résultats, nous conservons les mêmes variables explicatives et le même ensemble de validation que pour la régression linéaire globale filtrée.

L'entraînement repose sur un ensemble de 100 arbres de décision, dont la profondeur et le nombre minimal d'observations par feuille sont limités afin de maîtriser la complexité du modèle.

Les prédictions obtenues seront évaluées à l'aide des métriques MAE et RMSE, puis comparées aux résultats des expériences précédentes.

Cette première configuration constitue un point de départ expérimental et ne fait pas encore l'objet d'une optimisation des hyperparamètres.

In [210]:
# Entraînement de la Random Forest
random_forest_pipeline.fit(
    X_train_filtered,
    y_train_filtered,
)

# Prédictions sur l'ensemble de validation
y_pred_rf = random_forest_pipeline.predict(X_val)

# Calcul des métriques
rf_mae = mean_absolute_error(y_val, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_val, y_pred_rf))

print("Entraînement terminé !")
print("Nombre de prédictions :", len(y_pred_rf))
print(f"MAE  : {rf_mae:.2f} kW")
print(f"RMSE : {rf_rmse:.2f} kW")

Entraînement terminé !
Nombre de prédictions : 34188
MAE  : 6.21 kW
RMSE : 9.61 kW


#### E.3 - Comparaison avec les modèles précédents

In [211]:
comparison = pd.DataFrame(
    [
        {
            "modele": "Baseline hebdomadaire H-168",
            "MAE": 6.78,
            "RMSE": 10.48,
        },
        {
            "modele": "Régression linéaire globale filtrée",
            "MAE": linear_filtered_mae,
            "RMSE": linear_filtered_rmse,
        },
        {
            "modele": "Random Forest",
            "MAE": rf_mae,
            "RMSE": rf_rmse,
        },
    ]
)

display(comparison.sort_values("MAE").round(2))

,modele,MAE,RMSE
2,Random Forest,6.21,9.61
0,Baseline hebdomadaire H-168,6.78,10.48
1,Régression linéaire globale filtrée,7.04,10.87


### E.4 — Analyse des performances du modèle global par type de site

Notre premier modèle Random Forest global a obtenu une **MAE de 6,21 kW** et une **RMSE de 9,61 kW**, surpassant ainsi la baseline hebdomadaire sur l'ensemble de validation.

Toutefois, ces métriques globales ne permettent pas d'évaluer précisément la qualité des prévisions pour chaque catégorie de bâtiment.

Les profils de consommation peuvent varier considérablement entre les sites industriels, les bureaux, les datacenters, les commerces et les établissements hospitaliers.

Nous allons donc analyser les performances du modèle par type de site, en conservant les mêmes données de validation et en comparant les résultats à ceux de la baseline hebdomadaire H-168.

Cette analyse permettra d'identifier les catégories pour lesquelles le modèle global est déjà satisfaisant et celles qui pourraient bénéficier d'un entraînement spécialisé.

In [212]:
# Construction du dataset d'évaluation
df_rf_evaluation = df_val[
    [
        "site_id",
        "site_type",
        "horizon",
        "target_consumption_kw",
        "target_lag_168h",
    ]
].copy()

# Ajout des prédictions du modèle global
df_rf_evaluation["prediction_rf"] = y_pred_rf

# Vérification de la disponibilité des références hebdomadaires
assert df_rf_evaluation["target_lag_168h"].notna().all()

# Calcul des métriques par type de site
results_by_type = []

for site_type, group in df_rf_evaluation.groupby("site_type"):
    y_true = group["target_consumption_kw"]

    rf_predictions = group["prediction_rf"]
    weekly_predictions = group["target_lag_168h"]

    results_by_type.append(
        {
            "site_type": site_type,
            "observations": len(group),
            "MAE_RF": mean_absolute_error(y_true, rf_predictions),
            "RMSE_RF": np.sqrt(mean_squared_error(y_true, rf_predictions)),
            "MAE_H168": mean_absolute_error(y_true, weekly_predictions),
            "RMSE_H168": np.sqrt(mean_squared_error(y_true, weekly_predictions)),
        }
    )

df_results_by_type = pd.DataFrame(results_by_type)

# Gain relatif : positif si la Random Forest améliore la baseline
df_results_by_type["gain_MAE_pct"] = (
    (df_results_by_type["MAE_H168"] - df_results_by_type["MAE_RF"])
    / df_results_by_type["MAE_H168"]
    * 100
)

display(df_results_by_type.sort_values("gain_MAE_pct", ascending=False).round(2))

,site_type,observations,MAE_RF,RMSE_RF,MAE_H168,RMSE_H168,gain_MAE_pct
3,office,9768,1.38,1.92,1.63,2.25,15.25
0,datacenter,4884,11.82,14.68,13.42,16.57,11.90
2,hospital,4884,7.18,9.02,7.74,9.87,7.25
1,factory,9768,9.05,12.63,9.72,13.47,6.93
4,retail,4884,3.61,4.90,3.62,4.89,0.10


## F — Expérimentation de modèles spécialisés par type de site

### F.1 — Objectifs et méthodologie

Les expérimentations précédentes ont démontré qu'un modèle Random Forest global pouvait améliorer les prévisions de consommation énergétique sur l'ensemble des catégories de sites.

Toutefois, les performances observées varient selon les profils énergétiques, notamment entre les bureaux, les usines, les datacenters, les commerces et les établissements hospitaliers.

Nous souhaitons désormais déterminer si l'entraînement de modèles spécialisés par type de site permet d'améliorer les résultats obtenus.

Pour cette expérience, nous allons entraîner **cinq modèles Random Forest indépendants**, chacun utilisant exclusivement les observations correspondant à une catégorie de bâtiment.

Afin de garantir une comparaison cohérente avec notre modèle global, nous conservons :

- La même séparation chronologique entre entraînement et validation.
- Les mêmes variables explicatives et le même pipeline de prétraitement.
- Les mêmes hyperparamètres Random Forest.
- La même stratégie d'exclusion des observations sans référence hebdomadaire.
- Les mêmes métriques d'évaluation : MAE et RMSE.

L'objectif est d'isoler l'effet de la spécialisation des modèles, sans introduire simultanément d'autres modifications.

Cette expérimentation permettra de déterminer si les gains éventuels justifient la complexité supplémentaire liée à l'entraînement, au déploiement et à la maintenance de plusieurs modèles.

#### F.2 — Constitution des ensembles spécialisés

In [213]:
# Répartition des observations par type de site

train_distribution = X_train_filtered["site_type"].value_counts().rename("train_samples")

val_distribution = X_val["site_type"].value_counts().rename("val_samples")

df_distribution = (
    pd.concat(
        [train_distribution, val_distribution],
        axis=1,
    )
    .fillna(0)
    .astype(int)
)

df_distribution["total_samples"] = df_distribution["train_samples"] + df_distribution["val_samples"]

display(df_distribution.sort_index())

print("Total entraînement :", df_distribution["train_samples"].sum())
print("Total validation :", df_distribution["val_samples"].sum())

,train_samples,val_samples,total_samples
site_type,,,
datacenter,8064,4884,12948
factory,16128,9768,25896
hospital,8064,4884,12948
office,16128,9768,25896
retail,8064,4884,12948


Total entraînement : 56448
Total validation : 34188


#### F.3 — Entraînement des modèles Random Forest spécialisés

Nous procédons à l'entraînement de cinq modèles Random Forest, chacun spécialisé dans une catégorie de bâtiment.

Chaque modèle est entraîné exclusivement sur les observations de sa catégorie, tout en conservant les mêmes variables explicatives, les mêmes transformations et les mêmes hyperparamètres que le modèle global.

Cette approche permet d'étudier si la spécialisation améliore la capacité des modèles à représenter les comportements énergétiques propres à chaque catégorie.

Les modèles sont entraînés et évalués indépendamment. Leurs résultats seront ensuite comparés à ceux du modèle global et de la baseline hebdomadaire, sur les mêmes observations de validation.

Les résultats permettront de déterminer si les éventuels gains de précision justifient la complexité supplémentaire d'une architecture reposant sur plusieurs modèles.

In [214]:
# Stockage des modèles et des résultats
specialized_models = {}
specialized_results = []
specialized_predictions = {}

for site_type in sorted(X_train_filtered["site_type"].unique()):
    print(f"Entraînement du modèle : {site_type}")

    # Sélection des observations du type de site
    train_mask = X_train_filtered["site_type"] == site_type
    val_mask = X_val["site_type"] == site_type

    X_train_type = X_train_filtered.loc[train_mask]
    y_train_type = y_train_filtered.loc[train_mask]

    X_val_type = X_val.loc[val_mask]
    y_val_type = y_val.loc[val_mask]

    # Nouveau pipeline indépendant
    model = clone(random_forest_pipeline)

    # Entraînement
    model.fit(X_train_type, y_train_type)

    # Prédictions
    predictions = model.predict(X_val_type)

    # Évaluation
    mae = mean_absolute_error(y_val_type, predictions)
    rmse = np.sqrt(mean_squared_error(y_val_type, predictions))

    # Conservation des résultats
    specialized_models[site_type] = model
    specialized_predictions[site_type] = pd.Series(
        predictions,
        index=X_val_type.index,
    )

    specialized_results.append(
        {
            "site_type": site_type,
            "observations": len(X_val_type),
            "MAE_specialized": mae,
            "RMSE_specialized": rmse,
        }
    )

    print(f"  MAE  : {mae:.2f} kW")
    print(f"  RMSE : {rmse:.2f} kW\n")

df_specialized_results = pd.DataFrame(specialized_results)

display(df_specialized_results.round(2))

Entraînement du modèle : datacenter
  MAE  : 12.53 kW
  RMSE : 15.18 kW

Entraînement du modèle : factory
  MAE  : 8.99 kW
  RMSE : 12.40 kW

Entraînement du modèle : hospital
  MAE  : 7.27 kW
  RMSE : 9.07 kW

Entraînement du modèle : office
  MAE  : 1.40 kW
  RMSE : 1.91 kW

Entraînement du modèle : retail
  MAE  : 3.71 kW
  RMSE : 5.07 kW



,site_type,observations,MAE_specialized,RMSE_specialized
0,datacenter,4884,12.53,15.18
1,factory,9768,8.99,12.40
2,hospital,4884,7.27,9.07
3,office,9768,1.40,1.91
4,retail,4884,3.71,5.07


#### F.4 — Comparaison globale et spécialisée

In [215]:
# Rapprochement des résultats par catégorie
df_architecture_comparison = df_results_by_type.merge(
    df_specialized_results,
    on=["site_type", "observations"],
)

# Gain de la spécialisation par rapport au modèle global
df_architecture_comparison["gain_specialization_pct"] = (
    (df_architecture_comparison["MAE_RF"] - df_architecture_comparison["MAE_specialized"])
    / df_architecture_comparison["MAE_RF"]
    * 100
)

display(
    df_architecture_comparison[
        [
            "site_type",
            "MAE_RF",
            "MAE_specialized",
            "RMSE_RF",
            "RMSE_specialized",
            "gain_specialization_pct",
        ]
    ].round(2)
)

,site_type,MAE_RF,MAE_specialized,RMSE_RF,RMSE_specialized,gain_specialization_pct
0,datacenter,11.82,12.53,14.68,15.18,-5.99
1,factory,9.05,8.99,12.63,12.40,0.62
2,hospital,7.18,7.27,9.02,9.07,-1.34
3,office,1.38,1.40,1.92,1.91,-1.04
4,retail,3.61,3.71,4.90,5.07,-2.70


In [216]:
# Reconstitution des prédictions spécialisées
# dans l'ordre des observations de validation
y_pred_specialized = pd.concat(specialized_predictions.values()).reindex(X_val.index)

assert y_pred_specialized.notna().all()

# Métriques globales des modèles spécialisés
specialized_mae = mean_absolute_error(
    y_val,
    y_pred_specialized,
)

specialized_rmse = np.sqrt(mean_squared_error(y_val, y_pred_specialized))

print(f"Random Forest globale     — MAE : {rf_mae:.2f} kW")
print(f"Random Forest spécialisée — MAE : {specialized_mae:.2f} kW")

print(f"Random Forest globale     — RMSE : {rf_rmse:.2f} kW")
print(f"Random Forest spécialisée — RMSE : {specialized_rmse:.2f} kW")

Random Forest globale     — MAE : 6.21 kW
Random Forest spécialisée — MAE : 6.33 kW
Random Forest globale     — RMSE : 9.61 kW
Random Forest spécialisée — RMSE : 9.66 kW



## G — Conclusion et bilan de l'expérimentation

Cette première campagne d'entraînement avait pour objectif d'évaluer la pertinence de différentes approches de Machine Learning pour la prévision de consommation énergétique à court terme, sur des horizons allant de H+1 à H+24.

L'expérimentation a été réalisée à partir de trente jours de readings EnerVision, agrégés à une granularité horaire et transformés en données supervisées.

### G.1 — Synthèse des résultats

Plusieurs approches ont été étudiées : une régression linéaire, une Random Forest globale entraînée sur l'ensemble des bâtiments et des Random Forest spécialisées par catégorie de bâtiment.

Les résultats obtenus sur l'ensemble de validation sont les suivants :

| Modèle | MAE (kW) | RMSE (kW) |
|---|---:|---:|
| Baseline journalière H-24 | 36,49 | 114,25 |
| Baseline hebdomadaire H-168 | 6,78 | 10,48 |
| Régression linéaire globale (données filtrées) | 7,04 | 10,87 |
| Random Forest globale | **6,21** | **9,61** |
| Random Forest spécialisées | 6,33 | 9,66 |

La Random Forest globale obtient les meilleurs résultats, avec une réduction de la MAE d'environ 8,4 % par rapport à la baseline hebdomadaire.

Les modèles spécialisés ne présentent pas d'amélioration globale par rapport au modèle unique. Leur intérêt apparaît donc limité dans le contexte de cette première expérimentation, d'autant qu'ils impliqueraient une complexité supplémentaire en matière d'entraînement, de déploiement et de maintenance.

### G.2 — Enseignements techniques

Cette expérimentation a permis de mettre en évidence plusieurs points importants.

**Importance des caractéristiques historiques**

La consommation observée à la même heure la semaine précédente constitue une variable particulièrement pertinente pour la prévision. La très bonne performance de la baseline H-168 suggère une forte régularité hebdomadaire dans les données étudiées.

**Impact du traitement des valeurs manquantes**

L'imputation médiane de la référence hebdomadaire a initialement entraîné une dégradation importante des performances de la régression linéaire. Le filtrage des observations ne disposant pas de cette référence a permis de retrouver des résultats nettement plus cohérents.

Ce constat souligne la nécessité de définir une stratégie explicite de gestion des historiques incomplets, notamment pour les futures prévisions en production.

**Intérêt d'un modèle global**

La Random Forest globale obtient des résultats légèrement supérieurs aux modèles spécialisés. Cette observation favorise provisoirement une architecture reposant sur un modèle unique, capable de prendre en compte l'identité et la catégorie des bâtiments.

Cette orientation reste toutefois à confirmer sur un historique plus représentatif.

### G.3 — Limites de l'expérimentation

Malgré des performances encourageantes, plusieurs limites doivent être prises en compte.

- **Historique restreint :** les trente jours de readings ne permettent pas d'observer les variations saisonnières ou les évolutions de consommation sur plusieurs mois.
- **Régularité des données :** les très faibles erreurs de la baseline hebdomadaire suggèrent des comportements particulièrement répétitifs, qui peuvent faciliter artificiellement la prévision.
- **Nombre limité de bâtiments :** l'expérimentation repose sur sept sites, dont certaines catégories ne sont représentées que par un seul bâtiment.
- **Validation temporelle limitée :** les performances ont été mesurées sur une période de validation chronologique, sans ensemble de test final indépendant.
- **Historique incomplet :** la stratégie retenue nécessite une référence hebdomadaire disponible et ne couvre donc pas encore toutes les situations d'inférence.

Ces limites empêchent de généraliser directement les performances obtenues à d'autres périodes, bâtiments ou sources de données.

### G.4 — Perspectives et transition vers l'étude historique

Cette première expérimentation valide la faisabilité technique d'un système de prévision énergétique multi-horizon fondé sur une Random Forest.

Elle met également en évidence l'intérêt d'approfondir l'étude de la représentativité des données d'entraînement.

Une seconde campagne expérimentale a ainsi été menée sur un jeu de données historique couvrant les années 2023 et 2024, afin de confronter notre approche à une plus grande diversité temporelle.

Cette nouvelle étude a notamment permis de confirmer l'intérêt de la Random Forest globale sur un ensemble de test indépendant, avec une réduction de la MAE de 22,18 % par rapport à la baseline hebdomadaire.

Toutefois, les différences entre les deux jeux de données ne permettent pas d'attribuer directement cette amélioration à la seule durée de l'historique.

Une dernière expérimentation sera donc consacrée à l'influence de la fenêtre d'entraînement, en comparant plusieurs durées d'historique sur une période d'évaluation commune.

Les résultats de ces travaux serviront à orienter la conception du futur pipeline ML/MLOps, notamment concernant la constitution des datasets, le traitement des valeurs manquantes, le réentraînement et l'industrialisation des modèles.
